## Tâche 6 — Mon pipeline v2

La v1 prend 15 colonnes et s'arrête là. Je pars des mêmes données brutes, j'ajoute
des variables que la base ne contient pas, et je compare trois familles de modèles
sur le même jeu.

Tout se passe dans le pipeline. Rien n'est calculé avant, sinon train.py ne pourrait
pas rejouer la recette sur des données neuves.

In [20]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# le notebook tourne depuis notebooks/, la racine du projet est au-dessus
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

query = text("""
    SELECT r.*, h.code AS hotel
    FROM reservations r
    JOIN hotels h ON r.hotel_id = h.hotel_id
""")
df = pd.read_sql(query, engine, parse_dates=["arrival_date", "reservation_status_date"])

TEST_MONTHS = 4

# le test commence au 1er jour du 4e mois avant la fin de l'historique
debut_test = (df["arrival_date"].max().to_period("M") - (TEST_MONTHS - 1)).to_timestamp()

train = df[df["arrival_date"] < debut_test].copy()
test = df[df["arrival_date"] >= debut_test].copy()

print(f"train : {len(train)} | test : {len(test)}")

train : 58792 | test : 19911


Cellule 1 — les colonnes et les nouvelles variables

In [14]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# les 6 catégorielles de la v1, plus 3 que la v1 n'utilisait pas
CATEGORICAL = ["hotel", "arrival_date_month", "market_segment", "deposit_type",
               "customer_type", "country", "meal", "distribution_channel",
               "reserved_room_type"]

# les 9 numériques de la v1, plus 4 colonnes brutes et mes 4 variables créées
NUMERICAL = ["lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
             "adults", "children", "babies", "adr", "previous_cancellations",
             "previous_bookings_not_canceled", "total_of_special_requests",
             "required_car_parking_spaces", "is_repeated_guest", "days_in_waiting_list",
             "duree_sejour", "taille_groupe", "avec_enfants", "taux_annulation_passe"]


def ajouter_variables(donnees):
    d = donnees.copy()  # on ne modifie jamais le DataFrame reçu

    enfants = d["children"].fillna(0) + d["babies"]  # children a des valeurs manquantes

    d["duree_sejour"] = d["stays_in_weekend_nights"] + d["stays_in_week_nights"]
    d["taille_groupe"] = d["adults"] + enfants
    d["avec_enfants"] = (enfants > 0).astype(int)  # 0/1, le modèle n'accepte pas les booléens partout

    historique = d["previous_cancellations"] + d["previous_bookings_not_canceled"]
    # part des annulations dans le passé du client, 0 si on ne le connaît pas
    d["taux_annulation_passe"] = np.where(
        historique > 0,
        d["previous_cancellations"] / historique.replace(0, np.nan),  # évite la division par zéro
        0.0,
    )

    return d[CATEGORICAL + NUMERICAL]  # on ne laisse passer que ce qui entre dans le modèle

Cellule 2 — le pipeline


In [15]:
def build_pipeline_v2(classifier, mise_a_l_echelle=False):
    etapes_num = [("impute", SimpleImputer(strategy="median"))]  # médiane : robuste aux valeurs extrêmes
    if mise_a_l_echelle:
        etapes_num.append(("scale", StandardScaler()))  # utile pour la régression logistique, inutile aux arbres

    preprocessing = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                # min_frequency=50 : les modalités rares sont regroupées, sinon country explose en 148 colonnes
                ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)),
            ]), CATEGORICAL),
            ("num", Pipeline(etapes_num), NUMERICAL),
        ],
        sparse_threshold=0,  # sortie dense, les forêts n'aiment pas les matrices creuses
    )

    return Pipeline([
        ("variables", FunctionTransformer(ajouter_variables)),  # données brutes -> variables enrichies
        ("preprocessing", preprocessing),
        ("classifier", classifier),
    ])

Cellule 3 — comparer les familles

In [16]:
X_train = train.drop(columns=["is_canceled"])  # on passe les colonnes brutes, le pipeline fait le tri
y_train = train["is_canceled"]

METRIQUES = ["f1", "precision", "recall", "accuracy", "roc_auc"]

familles = {
    "RandomForest": (RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                            random_state=42, n_jobs=-1), False),
    "HistGradientBoosting": (HistGradientBoostingClassifier(max_iter=200, random_state=42), False),
    "RegressionLogistique": (LogisticRegression(max_iter=1000, random_state=42), True),
}

lignes = []
for nom, (modele, echelle) in familles.items():
    scores = cross_validate(build_pipeline_v2(modele, echelle), X_train, y_train,
                            cv=5, scoring=METRIQUES, return_train_score=True, n_jobs=-1)
    ligne = {"famille": nom}
    for m in METRIQUES:
        ligne[m] = round(scores[f"test_{m}"].mean(), 4)
    ligne["ecart_train"] = round(scores["train_f1"].mean() - scores["test_f1"].mean(), 4)
    lignes.append(ligne)

pd.DataFrame(lignes)

,famille,f1,precision,recall,accuracy,roc_auc,ecart_train
0,RandomForest,0.7485,0.8523,0.6853,0.8393,0.9079,0.1723
1,HistGradientBoosting,0.7121,0.7893,0.6902,0.8017,0.8856,0.1682
2,RegressionLogistique,0.6679,0.7682,0.6230,0.7779,0.8778,0.0827


### Ce que je retiens

La v2 passe de 0,70 à 0,75 de F1. Mais le gain ne vient pas d'où je croyais.

En remplaçant juste le boosting par une forêt aléatoire, sur les 15 variables
de la v1 sans rien toucher d'autre, je passe déjà à 0,74. Mes quatre variables
ajoutent 0,7 point par-dessus. C'est le changement de famille qui fait le travail.

J'ai aussi retiré trois colonnes. arrival_date_week_number faisait perdre 3 points :
le modèle apprend des semaines de 2015 et 2016 qui ne reviennent pas. assigned_room_type
et booking_changes sont des fuites, comme reservation_status en tâche 4 : au moment où
je prédis, à la réservation, ces colonnes ne sont pas encore remplies.

Il reste 0,17 d'écart entre entraînement et validation. La forêt apprend trop ses
données. 

## Tâche 7 — Valider et régler

La forêt a 0,17 d'écart entre entraînement et validation. Elle apprend ses données
par cœur. Je cherche les réglages qui la contraignent.

Deux leviers : min_samples_leaf, le nombre minimum de réservations dans une feuille,
et max_features, la part des colonnes que chaque arbre a le droit de regarder.
Plus on les serre, moins le modèle peut coller aux cas particuliers.

Je trace toute la grille, pas seulement le gagnant. C'est la colonne mean_train_score
qui raconte le surapprentissage.

Cellule 1 — grille large

In [17]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

grille = {
    "classifier__min_samples_leaf": [1, 2, 5, 10, 20],   # 1 = une feuille par cas extrême
    "classifier__max_features": ["sqrt", 0.3, 0.5],      # sqrt = ~5 colonnes sur 26
}

recherche = GridSearchCV(
    # n_jobs=1 sur la forêt : c'est GridSearchCV qui parallélise, pas elle
    build_pipeline_v2(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)),
    grille,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True,  # indispensable pour voir l'écart, pas activé par défaut
)
recherche.fit(X_train, y_train)

resultats = pd.DataFrame(recherche.cv_results_)
resultats["ecart"] = resultats["mean_train_score"] - resultats["mean_test_score"]

resultats[["param_classifier__min_samples_leaf", "param_classifier__max_features",
           "mean_train_score", "mean_test_score", "std_test_score", "ecart"]] \
    .sort_values("mean_test_score", ascending=False).round(4)

,param_classifier__min_samples_leaf,param_classifier__max_features,mean_train_score,mean_test_score,std_test_score,ecart
14,20,0.5,0.8650,0.7624,0.0703,0.1026
8,10,0.3,0.8799,0.7612,0.0754,0.1187
9,20,0.3,0.8599,0.7599,0.0729,0.1000
13,10,0.5,0.8877,0.7586,0.0760,0.1291
7,5,0.3,0.9080,0.7575,0.0803,0.1505
12,5,0.5,0.9177,0.7556,0.0794,0.1621
6,2,0.3,0.9605,0.7526,0.0842,0.2079
2,5,sqrt,0.8719,0.7501,0.0867,0.1218
11,2,0.5,0.9680,0.7492,0.0861,0.2188
1,2,sqrt,0.9209,0.7485,0.0892,0.1723


Cellule 2 — grille affinée

In [18]:
grille_2 = {
    "classifier__min_samples_leaf": [20, 50, 100, 150, 250],  # on pousse au-delà de la borne
    "classifier__max_features": [0.3, 0.5, 0.7],
}

recherche_2 = GridSearchCV(
    build_pipeline_v2(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1)),
    grille_2, cv=5, scoring="f1", n_jobs=-1, return_train_score=True,
)
recherche_2.fit(X_train, y_train)

resultats_2 = pd.DataFrame(recherche_2.cv_results_)
resultats_2["ecart"] = resultats_2["mean_train_score"] - resultats_2["mean_test_score"]

print(recherche_2.best_params_, round(recherche_2.best_score_, 4))
resultats_2[["param_classifier__min_samples_leaf", "param_classifier__max_features",
             "mean_train_score", "mean_test_score", "ecart"]] \
    .sort_values("mean_test_score", ascending=False).round(4)

{'classifier__max_features': 0.7, 'classifier__min_samples_leaf': 100} 0.7679


,param_classifier__min_samples_leaf,param_classifier__max_features,mean_train_score,mean_test_score,ecart
12,100,0.7,0.8303,0.7679,0.0624
7,100,0.5,0.8305,0.7676,0.0629
13,150,0.7,0.8199,0.7645,0.0553
1,50,0.3,0.8441,0.7639,0.0802
5,20,0.5,0.8650,0.7624,0.1026
2,100,0.3,0.8268,0.7624,0.0644
6,50,0.5,0.8473,0.7621,0.0852
8,150,0.5,0.8201,0.7619,0.0583
3,150,0.3,0.8166,0.7617,0.0549
10,20,0.7,0.8673,0.7609,0.1064


Cellule 3 — valider le modèle retenu

In [19]:
foret_reglee = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=100,   # une feuille couvre au moins 100 réservations
    max_features=0.7,       # chaque arbre voit 70 % des colonnes
    random_state=42,
    n_jobs=-1,
)

# découpage classique : les plis sont mélangés dans le temps
scores = cross_validate(build_pipeline_v2(foret_reglee), X_train, y_train,
                        cv=5, scoring=METRIQUES, return_train_score=True, n_jobs=-1)

# découpage temporel : on n'entraîne que sur le passé de chaque pli
train_trie = train.sort_values("arrival_date")
scores_ts = cross_validate(build_pipeline_v2(foret_reglee),
                           train_trie.drop(columns=["is_canceled"]), train_trie["is_canceled"],
                           cv=TimeSeriesSplit(n_splits=5), scoring=METRIQUES,
                           return_train_score=True, n_jobs=-1)

for m in METRIQUES:
    t, v = scores[f"train_{m}"].mean(), scores[f"test_{m}"].mean()
    ts = scores_ts[f"test_{m}"].mean()
    print(f"{m:10} train {t:.4f} | validation {v:.4f} | ecart {t-v:+.4f} | temporel {ts:.4f}")

f1         train 0.8303 | validation 0.7679 | ecart +0.0624 | temporel 0.7416
precision  train 0.8784 | validation 0.8545 | ecart +0.0239 | temporel 0.8806
recall     train 0.7873 | validation 0.7024 | ecart +0.0849 | temporel 0.6439
accuracy   train 0.8855 | validation 0.8514 | ecart +0.0341 | temporel 0.8483
roc_auc    train 0.9532 | validation 0.9151 | ecart +0.0381 | temporel 0.8870


### Ce que je retiens

Deux grilles. La première s'arrête sur sa borne, min_samples_leaf=20, donc je
l'élargis. La deuxième trouve le vrai maximum à 100, et ça redescend après.

Le réglage gagne 2 points de F1, 0,7485 à 0,7679. Mais le vrai résultat est ailleurs :
l'écart entre entraînement et validation tombe de 0,17 à 0,06. Le modèle arrête
d'apprendre ses lignes par cœur.

J'ai réglé le boosting aussi, pour être sûr de mon choix de famille. Il plafonne
à 0,7327. La forêt reste devant.

En découpage temporel, je descends à 0,74 et le rappel à 0,64. C'est l'estimation
la plus proche de ce que le test va donner.